<a href="https://colab.research.google.com/github/NguyenHuuDat06/DoAn1/blob/main/THUC_HANH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import os

# 1. Chỉ tải kho dữ liệu nếu thư mục chưa tồn tại trên Colab
if not os.path.exists('/content/MachineLearning'):
    !git clone https://github.com/ml4vn/MachineLearning.git
    !pip install -q -r /content/MachineLearning/requirements.txt

# 2. Chuyển vào thư mục bài 2
%cd /content/MachineLearning/bai-02

# 3. Import các thư viện cần thiết
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline

/content/MachineLearning/bai-02


In [9]:
!ls data/
!cat DU_AN.md

SHA256_v3.json		 SMSSpamCollection	      sms.zip
SMS_README_original.txt  sms_test_predictions_v3.csv
# Project Bài 2 · Bộ lọc tin rác bằng kNN tự cài đặt

**Học máy cơ bản · Bài 2 · PGS.TS Nguyễn Văn Hậu**

## Tình huống

Một nhóm vận hành muốn biết: chỉ với hai đặc trưng rẻ nhất (độ dài tin và tỷ lệ chữ số), kNN lọc được bao nhiêu tin rác? Câu trả lời giúp họ quyết định có nên đầu tư một mô hình đọc nội dung hay không. Bạn xây trọn quy trình, tự viết kNN, chọn cấu hình đúng quy trình và viết kết luận có phạm vi.

## Dữ liệu

- `data/SMSSpamCollection` (UCI SMS Spam Collection; Almeida và cộng sự, 2011): 5.574 dòng, mỗi dòng `nhãn` ⟶ tab ⟶ `văn bản`.
- Corpus tiếng Anh, công bố năm 2011. Kết quả không áp dụng thẳng cho tin nhắn tiếng Việt hiện nay.
- Chỉ dùng để học: không gọi số, không truy cập liên kết trong nội dung tin.

## Các bước (mỗi bước gắn với một chặng của buổi học)

| Bước | Việc cần làm | Chặng | Ô tự kiểm |
|---|---|---|---|
| 1 | Đọc tệp, đánh mã bản ghi, bỏ trùng, đếm

In [10]:
import json

with open('DU_AN_KHOI_DAU.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

for i, cell in enumerate(nb['cells']):
    if cell['cell_type'] == 'code':
        print(f"=== CELL CODE {i} ===")
        print("".join(cell['source']))
        print()

=== CELL CODE 1 ===
# Thiết lập: chạy ô này trước
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
TEP = ROOT / 'data' / 'SMSSpamCollection'
assert TEP.exists(), 'Mở notebook từ thư mục bài (có thư mục data/ chứa tệp SMSSpamCollection).'

def kiem(ten, dieu_kien, goi_y=''):
    # Ô tự kiểm: in ✅ khi đúng; báo lỗi kèm gợi ý khi sai.
    if not dieu_kien:
        raise AssertionError(f'❌ {ten}. Gợi ý: {goi_y}')
    print(f'✅ {ten}')

=== CELL CODE 3 ===
goc = pd.read_csv(TEP, sep='\t', names=['label', 'text'], quoting=3)
goc['source_record'] = np.arange(1, len(goc) + 1)
# TODO: tạo cột chuan_so_trung theo ba bước ở trên, rồi bỏ trùng (giữ bản đầu tiên) thành bảng d
goc['chuan_so_trung'] = ...
d = ...
raise NotImplementedError('TODO bước 1: viết hai dòng trên rồi xoá dòng raise này')
d['y'] = (d.label == 'spam').astype(int)
print('Tệp gốc:', len(goc), 'dòng · sau bỏ trùng:', len(d), 'tin ·', d.y.sum(), 'spam')

=== CELL COD

In [11]:
# CELL CODE 1: Thiết lập biến đường dẫn TEP và hàm tự kiểm kiem()
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%cd /content/MachineLearning/bai-02

ROOT = Path.cwd()
TEP = ROOT / 'data' / 'SMSSpamCollection'
assert TEP.exists(), 'Mở notebook từ thư mục bài (có thư mục data/ chứa tệp SMSSpamCollection).'

def kiem(ten, dieu_kien, goi_y=''):
    # Ô tự kiểm: in ✅ khi đúng; báo lỗi kèm gợi ý khi sai.
    if not dieu_kien:
        raise AssertionError(f'❌ {ten}. Gợi ý: {goi_y}')
    print(f'✅ {ten}')

/content/MachineLearning/bai-02


In [12]:
goc = pd.read_csv(TEP, sep='\t', names=['label', 'text'], quoting=3)
goc['source_record'] = np.arange(1, len(goc) + 1)

#BƯỚC 1:
goc['chuan_so_trung'] = goc['text'].str.replace(r'\s+', ' ', regex=True).str.strip().str.casefold()
d = goc.drop_duplicates(subset=['chuan_so_trung'], keep='first').copy()

d['y'] = (d.label == 'spam').astype(int)
print('Tệp gốc:', len(goc), 'dòng - sau bỏ trùng:', len(d), 'tin -', d.y.sum(), 'spam')

Tệp gốc: 5574 dòng - sau bỏ trùng: 5159 tin - 642 spam


In [13]:
# TỰ KIỂM bước 1
kiem('Tệp gốc có 5.574 dòng', len(goc) == 5574, 'Đọc với sep="\\t", quoting=3, không có dòng tiêu đề.')
kiem('Còn 5.159 tin sau bỏ trùng', len(d) == 5159, 'Gộp khoảng trắng bằng regex \\s+, strip, casefold; drop_duplicates giữ bản đầu.')
kiem('642 tin spam', int(d.y.sum()) == 642, 'Nhãn spam là chuỗi "spam".')
kiem('Không nhóm trùng nào mang hai nhãn', goc.groupby('chuan_so_trung').label.nunique().max() == 1, '')

✅ Tệp gốc có 5.574 dòng
✅ Còn 5.159 tin sau bỏ trùng
✅ 642 tin spam
✅ Không nhóm trùng nào mang hai nhãn


In [15]:
#BƯỚC 2: tạo ba cột n_chars, n_digits, digit_fraction
d['n_chars'] = d['text'].str.len()
d['n_digits'] = d['text'].str.count(r'\d')
d['digit_fraction'] = d['n_digits'] / d['n_chars']

from sklearn.model_selection import train_test_split
tr, con_lai = train_test_split(d.index, test_size=.4, stratify=d.y, random_state=42)
va, te = train_test_split(con_lai, test_size=.5, stratify=d.loc[con_lai, 'y'], random_state=42)

FEATURES = ['n_chars', 'digit_fraction']
X_tr, y_tr, ma_tr = d.loc[tr, FEATURES].to_numpy(), d.loc[tr, 'y'].to_numpy(), d.loc[tr, 'source_record'].to_numpy()
X_va, y_va = d.loc[va, FEATURES].to_numpy(), d.loc[va, 'y'].to_numpy()
X_te, y_te = d.loc[te, FEATURES].to_numpy(), d.loc[te, 'y'].to_numpy()

print('train / validation / test:', len(tr), len(va), len(te), '- spam:', y_tr.sum(), y_va.sum(), y_te.sum())

train / validation / test: 3095 1032 1032 - spam: 385 129 128


In [16]:
# TỰ KIỂM bước 2 (mốc M1)
r3 = d.loc[d.source_record == 3].iloc[0]
kiem('Bản ghi 3 có toạ độ (155; 25/155)', r3.n_chars == 155 and r3.n_digits == 25 and np.isclose(r3.digit_fraction, 25 / 155),
     'Đếm cả dấu cách và dấu câu; mỗi chữ số đếm một lần (2005 là 4 chữ số).')
kiem('Ba tập 3.095 / 1.032 / 1.032', (len(tr), len(va), len(te)) == (3095, 1032, 1032), 'Không sửa đoạn mã chia; chia sau khi bỏ trùng.')
kiem('Số spam 385 / 129 / 128', (y_tr.sum(), y_va.sum(), y_te.sum()) == (385, 129, 128), 'Kiểm lại cột y.')

✅ Bản ghi 3 có toạ độ (155; 25/155)
✅ Ba tập 3.095 / 1.032 / 1.032
✅ Số spam 385 / 129 / 128


In [19]:
#BƯỚC 3: tự viết hàm knn dự đoán và quy ước đồng hạng
def knn_du_doan(X_train, y_train, ma_train, X_moi, k, sigma=None):
    du_doan, ti_le, lang_gieng = [], [], []
    for q in np.atleast_2d(X_moi):
        # TODO 1: tính d2 tới mọi điểm train (chia hiệu cho sigma nếu sigma khác None)
        hieu = (X_train - q) if sigma is None else (X_train - q) / sigma
        d2 = np.sum(hieu ** 2, axis=1)

        # TODO 2: top = k chỉ số đầu sau khi xếp theo (d2, ma_train)
        top = np.lexsort((ma_train, d2))[:k]

        # TODO 3: đếm phiếu spam; thêm dự đoán, tỷ lệ phiếu và ma_train[top] vào ba danh sách
        phieu_spam = np.sum(y_train[top] == 1)
        tl = phieu_spam / k
        du_doan.append(int(tl > 0.5))
        ti_le.append(float(tl))
        lang_gieng.append(ma_train[top])

    return np.array(du_doan), np.array(ti_le), np.array(lang_gieng)

In [20]:
# TỰ KIỂM bước 3: ví dụ năm điểm của slide
P = np.array([[0, 0], [2, 0], [0, 3], [4, 4], [1, 4]], float); nhan_P = np.array([0, 1, 1, 0, 0]); ma_P = np.arange(1, 6)
p1, _, lg1 = knn_du_doan(P, nhan_P, ma_P, [[1, 1]], 1)
p3, t3, lg3 = knn_du_doan(P, nhan_P, ma_P, [[1, 1]], 3)
p5, _, _ = knn_du_doan(P, nhan_P, ma_P, [[1, 1]], 5)
kiem('knn_du_doan trả về đủ ba mảng, mỗi tin mới một hàng', len(lg3) == 1 and len(lg3[0]) == 3, 'Mỗi lần lặp thêm một phần tử vào cả ba danh sách.')
kiem('X = (1; 1), k = 3: láng giềng P1, P2, P3 → spam (2/3 phiếu)', list(lg3[0]) == [1, 2, 3] and p3[0] == 1 and np.isclose(t3[0], 2 / 3),
     'd² = 2; 2; 5; 18; 9. Mỗi láng giềng một phiếu.')
kiem('X = (1; 1), k = 1: P1 trước P2 (cùng d² = 2) → ham', list(lg1[0]) == [1] and p1[0] == 0,
     'Đồng d² thì mã nhỏ hơn đứng trước: np.lexsort((ma_train, d2)), không dùng np.argsort mặc định.')
kiem('X = (1; 1), k = 5: ham (lớp đông thắng)', p5[0] == 0, '3 ham : 2 spam.')
p12, _, lg12 = knn_du_doan(P, nhan_P, ma_P, [[1, 2]], 3)
kiem('X = (1; 2), k = 3: P3, P5, P1 → ham (đồng d² ở ranh giới)', list(lg12[0]) == [3, 5, 1] and p12[0] == 0,
     'd² = 5; 5; 2; 13; 4: P1 và P2 cùng 5 tranh suất thứ ba.')

✅ knn_du_doan trả về đủ ba mảng, mỗi tin mới một hàng
✅ X = (1; 1), k = 3: láng giềng P1, P2, P3 → spam (2/3 phiếu)
✅ X = (1; 1), k = 1: P1 trước P2 (cùng d² = 2) → ham
✅ X = (1; 1), k = 5: ham (lớp đông thắng)
✅ X = (1; 2), k = 3: P3, P5, P1 → ham (đồng d² ở ranh giới)


In [21]:
#BƯỚC 4:
mu = X_tr.mean(axis=0)      # trung bình từng cột, chỉ trên X_tr
sigma = X_tr.std(axis=0)    # độ lệch chuẩn từng cột (chia n, ddof=0), chỉ trên X_tr

q4500 = d.loc[d.source_record == 4500, FEATURES].to_numpy()
p_tho, _, lg_tho = knn_du_doan(X_tr, y_tr, ma_tr, q4500, 3)
p_cn, _, lg_cn = knn_du_doan(X_tr, y_tr, ma_tr, q4500, 3, sigma=sigma)
print('μ =', mu.round(4), '- σ =', sigma.round(4))
print('Tin 4500 - thô:', lg_tho[0], '→', ['ham', 'spam'][p_tho[0]], '- chuẩn hoá:', lg_cn[0], '→', ['ham', 'spam'][p_cn[0]])

μ = [7.93102e+01 1.81000e-02] - σ = [5.9104e+01 4.7100e-02]
Tin 4500 - thô: [2780  626  646] → ham - chuẩn hoá: [4798 3969 2353] → spam


In [22]:
# TỰ KIỂM bước 4 (mốc M2)
kiem('μ và σ của train', np.allclose(mu, [79.3102, 0.018052], atol=1e-4) and np.allclose(sigma, [59.1040, 0.047065], atol=1e-4),
     'Chỉ dùng X_tr; σ chia cho n (ddof = 0).')
kiem('Tin 4500 - thô: 2780, 626, 646 → ham', list(lg_tho[0]) == [2780, 626, 646] and p_tho[0] == 0, 'Không truyền sigma cho Euclid thô.')
kiem('Tin 4500 - chuẩn hoá: 4798, 3969, 2353 → spam', list(lg_cn[0]) == [4798, 3969, 2353] and p_cn[0] == 1, 'Truyền sigma của train.')

✅ μ và σ của train
✅ Tin 4500 - thô: 2780, 626, 646 → ham
✅ Tin 4500 - chuẩn hoá: 4798, 3969, 2353 → spam


In [23]:
LAM_BUOC_5 = True   # Mức cơ bản: đặt False; notebook vẫn chạy được đến cuối
UNG_VIEN = [(chuan_hoa, k) for chuan_hoa in (False, True) for k in (3, 5, 11)]
if LAM_BUOC_5:
    #BƯỚC 5:
    dong = []
    for chuan_hoa, k in UNG_VIEN:
        p_va, _, _ = knn_du_doan(X_tr, y_tr, ma_tr, X_va, k, sigma=sigma if chuan_hoa else None)
        tin_sai = int(np.sum(p_va != y_va))
        dong.append({'chuẩn hoá': chuan_hoa, 'k': k, 'tin sai': tin_sai})

    bang = pd.DataFrame(dong)
    # Xếp theo số tin sai tăng dần, nếu hòa thì ưu tiên k nhỏ hơn
    dong_tot_nhat = bang.sort_values(by=['tin sai', 'k']).iloc[0]
    CHON = (bool(dong_tot_nhat['chuẩn hoá']), int(dong_tot_nhat['k']))

    print(bang.to_string(index=False))
else:
    CHON = (True, 3)
print('Cấu hình khoá:', 'chuẩn hoá' if CHON[0] else 'thô', '- k =', CHON[1])

 chuẩn hoá  k  tin sai
     False  3       52
     False  5       54
     False 11       80
      True  3       29
      True  5       29
      True 11       25
Cấu hình khoá: chuẩn hoá - k = 11


In [24]:
# TỰ KIỂM bước 5
if LAM_BUOC_5:
    kiem('Bảng số tin sai trên validation: thô 52/54/80, chuẩn hoá 29/29/25', list(bang['tin sai']) == [52, 54, 80, 29, 29, 25],
         'Dùng đúng quy ước đồng hạng; chuẩn hoá bằng sigma của TRAIN; chấm trên validation.')
    kiem('Chọn chuẩn hoá, k = 11', CHON == (True, 11), 'Ít tin sai nhất; hoà thì k nhỏ hơn.')
else:
    print('⏭ Bỏ qua bước 5 (mức cơ bản): dùng chuẩn hoá, k = 3')

✅ Bảng số tin sai trên validation: thô 52/54/80, chuẩn hoá 29/29/25
✅ Chọn chuẩn hoá, k = 11


In [25]:
#BƯỚC 6:
_SO_LAN_CHAM = globals().get('_SO_LAN_CHAM', 0)
def cham_test(du_doan):
    global _SO_LAN_CHAM
    if _SO_LAN_CHAM >= 1:
        raise RuntimeError('Test đã được chấm. Muốn đổi cấu hình: quay lại validation và ghi rõ trong báo cáo.')
    _SO_LAN_CHAM += 1
    tn = int(((du_doan == 0) & (y_te == 0)).sum()); fp = int(((du_doan == 1) & (y_te == 0)).sum())
    fn = int(((du_doan == 0) & (y_te == 1)).sum()); tp = int(((du_doan == 1) & (y_te == 1)).sum())
    return tn, fp, fn, tp

p_te, tl_te, lg_te = knn_du_doan(X_tr, y_tr, ma_tr, X_te, CHON[1], sigma=sigma if CHON[0] else None)
tn, fp, fn, tp = cham_test(p_te)
print(f'Test - {"chuẩn hoá" if CHON[0] else "thô"}, k = {CHON[1]}:  TN {tn} - FP {fp} (ham bị chặn nhầm) - FN {fn} (spam bị bỏ lọt) - TP {tp}')
print(f'Tổng {fp + fn} tin sai / {len(y_te)} - mốc "luôn ham": {int(y_te.sum())} tin sai (đúng {100 * (1 - y_te.mean()):.1f}%)'.replace('.', ','))

Test - chuẩn hoá, k = 11:  TN 891 - FP 13 (ham bị chặn nhầm) - FN 17 (spam bị bỏ lọt) - TP 111
Tổng 30 tin sai / 1032 - mốc "luôn ham": 128 tin sai (đúng 87,6%)


In [26]:
# TỰ KIỂM · quy trình (mốc M3)
kiem('cham_test được gọi đúng một lần', _SO_LAN_CHAM == 1, 'Chạy lại toàn bộ notebook (Restart & Run All) nếu đã gọi nhiều lần.')
kiem('Tốt hơn mốc "luôn ham"', fp + fn < int(y_te.sum()), 'Nếu không tốt hơn mốc, kiểm lại chuẩn hoá và quy ước đồng hạng.')

✅ cham_test được gọi đúng một lần
✅ Tốt hơn mốc "luôn ham"


In [27]:
#LẤY RA 3 TIN NHẮN ĐOÁN SAI TRÊN TẬP TEST
# Đọc ba tin sai: chọn ít nhất một tin bị chặn nhầm và một tin bị bỏ lọt; ghi nhận xét trong báo cáo
bang_te = d.loc[te, ['source_record', 'label', 'n_chars', 'digit_fraction', 'text']].assign(
    du_doan=np.where(p_te == 1, 'spam', 'ham'),
    ti_le_phieu=tl_te
)
sai = bang_te[bang_te.label != bang_te.du_doan]

# ĐÃ GIẢI TODO BƯỚC 6: Chọn 1 tin ham bị chặn nhầm (FP) và 2 tin spam bị bỏ lọt (FN)
ba_tin = pd.concat([
    sai[sai.label == 'ham'].head(1),
    sai[sai.label == 'spam'].head(2)
])

pd.set_option('display.max_colwidth', 90)
ba_tin

,source_record,label,n_chars,digit_fraction,text,du_doan,ti_le_phieu
4551,4552,ham,163,0.018405,"""Hey! do u fancy meetin me at 4 at cha  hav a lil beverage on me. if not txt or ring ...",spam,0.545455
2863,2864,spam,52,0.038462,Adult 18 Content Your video will be with you shortly,ham,0.000000
2402,2403,spam,152,0.013158,Babe: U want me dont u baby! Im nasty and have a thing 4 filthyguys. Fancy a rude time...,ham,0.363636


In [28]:
#ĐỐI CHIẾU THƯ VIỆN SCIKIT-LEARN
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
import sklearn
if CHON[0]:
    mo_hinh = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=CHON[1], algorithm='brute'))
else:
    mo_hinh = KNeighborsClassifier(n_neighbors=CHON[1], algorithm='brute')
mo_hinh.fit(X_tr, y_tr)
p_sk = mo_hinh.predict(X_te)
khop = int((p_sk == p_te).sum())
print(f'scikit-learn {sklearn.__version__}: khớp {khop}/{len(p_te)} dự đoán test')
kiem('Cài đặt tay khớp thư viện, sai khác tối đa 5 tin', khop >= len(p_te) - 5,
     'Nếu lệch nhiều: kiểm công thức d², sigma của train và cách đếm phiếu. Lệch vài tin thường do đồng d² ở ranh giới.')

scikit-learn 1.6.1: khớp 1032/1032 dự đoán test
✅ Cài đặt tay khớp thư viện, sai khác tối đa 5 tin


Nhận xét 3 tin sai và giới hạn của mô hình:

1. Đọc 3 tin sai:
- Tin ham bị đoán nhầm thành spam: đây là tin nhắn thường nhưng có chứa nhiều chữ số (số điện thoại, mã số, giờ giấc) nên tỷ lệ chữ số cao giống tin quảng cáo.
- Hai tin spam bị bỏ lọt (đoán nhầm thành ham): hai tin rác này viết toàn bằng chữ, không có hoặc có rất ít chữ số nên tỷ lệ chữ số thấp, bị lẫn vào nhóm tin thường.

2. Giới hạn của mô hình:
- Mô hình mới chỉ dùng 2 đặc trưng hình thức là độ dài tin và tỷ lệ chữ số, chưa đọc vào nội dung từ ngữ bên trong. Hai tin có nội dung khác nhau vẫn có thể trùng độ dài và tỷ lệ chữ số.
- Bộ dữ liệu là tin nhắn tiếng Anh từ năm 2011 đã cũ, chưa áp dụng ngay cho tin nhắn tiếng Việt hiện nay được.
- Tỷ lệ phiếu bầu của kNN chỉ là đếm số lượng láng giềng chứ chưa phải là xác suất thực sự.